In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as dt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial.distance import cdist
from sklearn.utils.estimator_checks import check_estimator
from config import DATA_DIR

In [ ]:
df_parkings = pd.read_parquet(DATA_DIR / 'parkings.parquet')

In [ ]:
df_parkings_availabilities = pd.read_parquet(DATA_DIR / 'parking_availabilities.parquet')

In [ ]:
df_sks_users = pd.read_parquet(DATA_DIR / 'sks_users.parquet')

In [ ]:
df_calendar = pd.read_parquet(DATA_DIR / 'calendar.parquet')

In [ ]:
df_calendar.head()

This is a new dataframe, it's crated based on university_calendar it holds type of day feature for each day in semester across 4 semesters (for now)

In [ ]:
CONST_FREQUENCY_MINUTES = 5
CONST_CONVERT_32_BIT_VALUES = True
CONST_SHIFT_GUARD_PERIODS = 15
CONST_CANNIVALISATION_GUARD_PERIODS = 15


Global constants used to manage tranfrom classes

In [ ]:
pd.set_option('display.max_columns',30)
pd.set_option('display.max_rows',100)

In [ ]:
df_parkings.head(10)

In [ ]:
df_min = df_parkings_availabilities.groupby('parking_id')['spaces_left'].max()
df_min.name = 'max_spaces_left'
df_parkings = df_parkings.merge(df_min, left_on='id', right_index=True)

print(df_parkings[['name', 'places', 'max_spaces_left']])

In [ ]:
# Fill missing values for "Polinka"
df_parkings.loc[df_parkings['name'] == "Polinka", ['open_hour', 'close_hour']] = "00:00:00"

In [ ]:
# df_min = df_parkings_availabilities.groupby('parking_id')['spaces_left'].min()
# df_min.name = 'min_spaces_left'
# df_parkings = df_parkings.merge(df_min, left_on='id', right_index=True)
# df_parkings['max_measured_spaces'] = df_parkings['max_spaces_left'] - df_parkings['min_spaces_left']

# print(df_parkings[['name', 'places', 'min_spaces_left', 'max_measured_spaces']])


Polinka has 69 spaces - NICE

<img src="locations.png" />

 I generated this map using streamlit it shows locations of parkings on the map, Red is for parkings blue for SKS

In [ ]:
# def calculate_distance(x1, y1, x2, y2):
#     return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2) #THIS DOES NOT WORK WITH DEGREES I HAVE BEEN LIED TO MY WHOLE LIFE 




# def calculate_distance(lat1, lon1, lat2, lon2):
#     R = 6371000  # Radius of Earth in meters
#     phi1, phi2 = np.radians(lat1), np.radians(lat2)
#     dphi = np.radians(lat2 - lat1)
#     dlambda = np.radians(lon2 - lon1)
    
#     a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
#     c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
#     return R * c

# sks_lat = 51.1087712 # taken from google maps
# sks_lan = 17.0574415

# df_unique = df_parkings.drop_duplicates(subset=['id'])
# closest_data = []
# for _, row in df_unique.iterrows():
#     distances = []
#     for _, row2 in df_unique.iterrows():
#         if row['id'] != row2['id']:
#             d = calculate_distance(row['geo_lat'], row['geo_lan'], 
#                                    row2['geo_lat'], row2['geo_lan'])
#             distances.append((d, row2['id']))
    
#     distances.sort() # Sorts by the first element (the distance)
#     distance_to_sks= calculate_distance(row['geo_lat'], row['geo_lan'], 
#                                    sks_lat, sks_lan)
#     closest_data.append({
#         'id': row['id'],
#         'closest_id': distances[0][1],
#         'second_closest_id': distances[1][1],
#         'third_closest_id': distances[2][1],
#         'fourth_closest_id': distances[3][1],
#         'distance_to_skf': round(distance_to_sks, 2)
#     })



# df_features = pd.DataFrame(closest_data)
# df_parkings = df_parkings.merge(df_features, on='id')
# df_parkings.head(10)

In [ ]:
import pandas as pd

# Manualy collected distances based on google maps
manual_data = [
    {'id': 7, 'd_to_7': 0,    'd_to_4': 2600, 'd_to_2': 2100, 'd_to_5': 2100, 'd_to_6': 2500, 'dist_to_sks': 2500},
    {'id': 4, 'd_to_7': 2600, 'd_to_4': 0,    'd_to_2': 800,  'd_to_5': 350,  'd_to_6': 1800, 'dist_to_sks': 42},
    {'id': 2, 'd_to_7': 2100, 'd_to_4': 500,  'd_to_2': 0,    'd_to_5': 650,  'd_to_6': 1800, 'dist_to_sks': 450},
    {'id': 5, 'd_to_7': 2100, 'd_to_4': 350,  'd_to_2': 650,  'd_to_5': 0,    'd_to_6': 2400, 'dist_to_sks': 400},
    {'id': 6, 'd_to_7': 2500, 'd_to_4': 1800, 'd_to_2': 1800, 'd_to_5': 2400, 'd_to_6': 0,    'dist_to_sks': 1500}
]

processed_rows = []

for entry in manual_data:
    current_id = entry['id']
    
    # Create a simple list of neighbors: (distance, neighbor_id)
    neighbors = []
    for key, dist in entry.items():
        if key.startswith('d_to_') and key != f'd_to_{current_id}':
            neighbor_id = int(key.split('_')[-1]) # Extract 4 from 'd_to_4'
            neighbors.append((dist, neighbor_id))
            
    # Sort closest to farthest
    neighbors.sort()
    
    # Build the simple row
    processed_rows.append({
        'id': current_id,
        'closest_id': neighbors[0][1],
        'second_closest_id': neighbors[1][1],
        'third_closest_id': neighbors[2][1],
        'fourth_closest_id': neighbors[3][1],
        'distance_to_sks': entry['dist_to_sks']
    })

df_features = pd.DataFrame(processed_rows)
df_parkings = df_parkings.merge(df_features, on='id', how='left')

# Check result
df_parkings.head()

Calcualting distance based on coordinates was outperformed by using goofle maps to get real distance estimates ( cars need to drive on road)

In [ ]:
df_parkings_availabilities = df_parkings_availabilities.sort_values(by=['measured_at'])
df_parkings_availabilities = df_parkings_availabilities.reset_index(drop=True)
df_parkings_availabilities.head(10)

In [ ]:
df_parkings_availabilities.info()

# Parking resampler
This class resamples the dataframe to ensure 5-minute intervals. This reduces complexity while keeping most of the information. Also, it's the basis for further imputation and merging with SKS and calendar dataframes. 

In [ ]:
agg_rules = {
        'spaces_left': 'mean',
        #'trend': 'mean', # this trend is weird.. i still have no idea how to use it seems semi random at times 
    }

In [ ]:
%cd ../../src

In [ ]:
%%writefile preprocessing/parking_resampler.py 
import pandas as pd
import numpy as np 
import datetime as dt
from sklearn.base import BaseEstimator, TransformerMixin
class ParkingResampler(BaseEstimator, TransformerMixin):
    """Custom transformer for resampling parking availability data.
    It takes a Dataframe and returns a resampled version of it based on the specified aggregation rules and frequency.
        Parameters:
        - agg_rules: A dictionary specifying the aggregation rules for each column (e.g., {'spaces
        _left': 'mean'}).
        - rule: A string representing the resampling frequency (default is '5min').
        - convert_to_32: A boolean indicating whether to convert integer and float columns to 32
        bit types to save memory (default is False).
        - copy: A boolean indicating whether to create a copy of the input DataFrame before transformation (default is True).
        - group_cols: A list of column names to group by before resampling (default is ['parking_id']).
    """
    def __init__(self, agg_rules, rule='5min', convert_to_32=False, copy = True, group_cols =['parking_id']):
        self.rule = rule
        self.convert_to_32 = convert_to_32
        self.copy = copy
        self.group_cols = group_cols
        self.agg_rules = agg_rules
        

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        # Ensure measured_at is the index for resampling
        if self.copy:
            X = X.copy()

        grouping_instructions = self.group_cols + [pd.Grouper(key='measured_at', freq=self.rule)]
        
        # Perform the grouping and aggregation in one go
        df_resampled = (
            X.groupby(grouping_instructions)
             .agg(self.agg_rules)
             .reset_index()
        )
        
        if self.convert_to_32:
            try:
                int_columns = df_resampled.select_dtypes(include=['int64']).columns
                df_resampled[int_columns] = df_resampled[int_columns].astype(np.int32)
                float_columns = df_resampled.select_dtypes(include=['float64']).columns
                df_resampled[float_columns] = df_resampled[float_columns].astype(np.float32)
            except Exception as e:
                print(f"Error occurred while converting data types: {e}")   
        return df_resampled

In [ ]:
from preprocessing.parking_resampler import ParkingResampler

In [ ]:
parking_resampler = ParkingResampler(convert_to_32=CONST_CONVERT_32_BIT_VALUES, agg_rules=agg_rules)
df_resampled = parking_resampler.fit_transform(df_parkings_availabilities)
df_resampled.head(10)

In [ ]:
df_resampled.info()


Function for easily plotting given value for parkings. 

In [ ]:
def plot_parking_trends(df_resampled, df_parkings, plotted_value='spaces_left'):
    """
        Plots the trends of a specified value (default is 'spaces_left') over time for each parking ID in the resampled DataFrame.
        Each parking ID gets its own subplot, and the title of each subplot includes the parking ID and its corresponding name from the df_parkings DataFrame.
        The x-axis is shared across all subplots for easier comparison.
        """
    fig = make_subplots(
        rows=df_resampled['parking_id'].nunique(), 
        cols=1, 
        shared_xaxes=True, 
        subplot_titles=[f"Parking ID: {pid} - {df_parkings[df_parkings['id'] == pid]['name'].iloc[0]}" for pid in df_resampled['parking_id'].unique()],
        vertical_spacing=0.05
    )

    for i, pid in enumerate(df_resampled['parking_id'].unique(), start=1):
        temp_fig = px.line(df_resampled[df_resampled['parking_id'] == pid], x='measured_at', y=plotted_value)
        for trace in temp_fig.data:
            fig.add_trace(trace, row=i, col=1)

    fig.update_layout(height=300*df_resampled['parking_id'].nunique(), showlegend=False, title_text=f"{plotted_value} Over Time for Each Parking ID")
    fig.show()


In [ ]:
plot_parking_trends(df_resampled, df_parkings, plotted_value='spaces_left')

In [ ]:
df_sks_users.head(10)

# Parking - SKS merger 
This class is used to combine parking and sks data. Also creates new feature based on parking's distance to SKS

In [ ]:
%%writefile preprocessing/parking_sks_merger.py 
import pandas as pd
import numpy as np 
from sklearn.base import BaseEstimator, TransformerMixin

class ParkingSKSMerger(BaseEstimator, TransformerMixin):
    """Merges parking data with SKS user data based on nearest timestamps and calculates a ratio feature. Data from SKS 
    is shifted by a specified lag to account for data propagation delays and to prevent data leakage.
     Parameters:
     - sks_df: A DataFrame containing SKS user data with 'external_timestamp'
     - parkings_df: A DataFrame containing parking data with 'id' and 'distance_to_sks' columns.
     - freq_minutes: The frequency in minutes for merging SKS data (default is 5 minutes).
     - tolerance: The maximum allowed time difference in minutes for merging SKS data (default is 10 minutes).
     - convert_to_32: A boolean indicating whether to convert certain columns to 32-bit types to save memory (default is False).
     - users_lag: The number of periods to shift the SKS user data to account for lag (default is 12, which corresponds to 1 hour if the frequency is 5 minutes).
     - copy: A boolean indicating whether to create a copy of the input DataFrame before transformation (default is True).
    """


    def __init__(self, sks_df, parkings_df, freq_minutes=5, tolerance=10, convert_to_32=False, users_lag=12, copy = True):
        self.sks_df = sks_df[['external_timestamp', 'active_users']].sort_values('external_timestamp')
        self.parkings_df = parkings_df[['id', 'distance_to_sks']].copy()
        self.freq_minutes = freq_minutes
        self.tolerance = tolerance
        self.convert_to_32 = convert_to_32
        self.users_lag = users_lag
        self.copy = copy

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = X.columns
        return self  

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        if self.copy: 
            X = X.copy()
            
        X = X.sort_values('measured_at')
        #check if columns required to merge exist 
        try:
            parkings_subset = self.parkings_df[['id']]
        except KeyError as e:
            raise KeyError(f"Missing required columns in parkings_df. {e}")

        try:
            parking_subset = X[['parking_id', 'measured_at']]
        except KeyError as e:
            raise KeyError(f"Missing required columns in input X. Expected 'parking_id' and 'measured_at'. Original error: {e}")

        
        X = pd.merge(X, self.parkings_df, left_on='parking_id', right_on='id', how='left')
        X = X.drop(columns=['id'])  # Drop the redundant 'id' column after merge

        sks_lagged = self.sks_df.copy()
        sks_lagged['active_users'] = sks_lagged['active_users'].shift(self.users_lag)

        try:
            sks_subset = sks_lagged[['external_timestamp']]
        except KeyError as e:
            raise KeyError(f"Missing required columns in SKS data. {e}")

        X = pd.merge_asof(
            X, 
            sks_lagged, 
            left_on='measured_at', 
            right_on='external_timestamp', 
            direction='nearest', 
            tolerance=pd.Timedelta(minutes=self.tolerance)
        )

        X['sks_dist_to_users_ratio'] = round(X['active_users'] * (100/(X['distance_to_sks'] + 1e-6)), 2)  # Add small constant to avoid division by zero
        
        # Drop the extra timestamp column 
        if 'external_timestamp' in X.columns:
            X = X.drop(columns=['external_timestamp'])
            
        if self.convert_to_32:
            try:
                X['active_users'] = X['active_users'].astype(np.float32) # temporary conver it to float 32 because float allows for NaN values which we have in this column due to the merge_asof and the tolerance"
                X['distance_to_sks'] = X['distance_to_sks'].astype(np.float32)
                X['sks_dist_to_users_ratio'] = X['sks_dist_to_users_ratio'].astype(np.float32)
            except Exception as e:
                print(f"Error occurred while converting data types: {e}")
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        return list(input_features) + ['active_users', 'distance_to_sks', 'sks_dist_to_users_ratio']

In [ ]:
from preprocessing.parking_sks_merger import ParkingSKSMerger

In [ ]:
parking_sks_merger = ParkingSKSMerger(sks_df=df_sks_users, parkings_df=df_parkings, freq_minutes=CONST_FREQUENCY_MINUTES, convert_to_32=CONST_CONVERT_32_BIT_VALUES)
df_merged = parking_sks_merger.fit_transform(df_resampled)
df_merged.head(10)

# Parking - Calendar merger
This is very similar to SKS merger here each timestap has it's datetype assigned
1. normal day
2. changed (example: sometimes univeristy changes monday in to friday)
3. session
4. free day (holidays and weekends)

In [ ]:
%%writefile preprocessing/parking_calendar_merger.py 
import pandas as pd
import numpy as np 
from sklearn.base import BaseEstimator, TransformerMixin

class ParkingCalendarMerger(BaseEstimator, TransformerMixin):
    """Merges parking data with calendar data based on nearest dates and adds a day type feature.
        Parameters:
        - calendar_df: A DataFrame containing calendar data with 'date' and 'day_type_id' columns.
        - freq_minutes: The frequency in minutes for merging calendar data (default is 5 minutes
        - convert_to_32: A boolean indicating whether to convert the 'day_type_id' column to 32-bit integer type to save memory (default is False).
        - copy: A boolean indicating whether to create a copy of the input DataFrame before transformation (default is True).
    """
    
    def __init__(self, calendar_df, freq_minutes=5, convert_to_32=False, copy = True):
        self.calendar_df = calendar_df[['date', 'day_type_id']].copy()
        self.freq_minutes = freq_minutes
        self.convert_to_32 = convert_to_32
        self.copy = copy

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = X.columns
        return self  

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        if self.copy:
            X = X.copy()
        #check if columns required to merge exist
        try: 
            parking_subset = X[['measured_at']]
        except KeyError as e:
            raise KeyError(f"Missing required columns in input X. Expected 'measured_at'. Original error: {e}")

        X['measured_at'] = pd.to_datetime(X['measured_at'])
        X = X.sort_values('measured_at')

        X['merge_date'] = X['measured_at'].dt.date
        #check if calendar_df has the required columns for merging
        try :
            calendar_subset = self.calendar_df[['date']]
        except KeyError as e:
            raise KeyError(f"Missing required columns in calendar_df. Expected 'date'. Original error: {e}")

        X = pd.merge(X, self.calendar_df, left_on='merge_date', right_on='date', how='left')

        X = X.drop(columns=['date', 'merge_date'])  # Drop the redundant 'date' and 'merge_date' columns after merge
            
        if self.convert_to_32:
            X['day_type_id'] = X['day_type_id'].astype(np.int32)
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        return list(input_features) + ['day_type_id']

In [ ]:
from preprocessing.parking_calendar_merger import ParkingCalendarMerger

In [ ]:
parking_claendar_merger = ParkingCalendarMerger(calendar_df=df_calendar, freq_minutes=CONST_FREQUENCY_MINUTES, convert_to_32=CONST_CONVERT_32_BIT_VALUES)
df_megred_calendar = parking_claendar_merger.fit_transform(df_merged)
df_megred_calendar['day_type_id'].value_counts()

# Imputer
This imputer prevents data leaks; it relies on lag features and ffill. This avoids peeking into the future.

In [ ]:
%%writefile preprocessing/parking_imputer.py 
import pandas as pd
import numpy as np 
from sklearn.base import BaseEstimator, TransformerMixin

class ParkingImputer(BaseEstimator, TransformerMixin):
    """Imputes missing values in parking data using a combination of forward-fill, seasonality-based filling, and median imputation.
    It also creates binary flag columns to indicate which values were imputed.
        Parameters:
        - cols_to_impute: A list of column names to impute (default is ['spaces_left']).
        - freq_minutes: The frequency in minutes for determining the forward-fill limit (default is 5 minutes).
        - ffill_limit: The maximum time in minutes to forward-fill missing values (default is 60 minutes).
        - convert_to_32: A boolean indicating whether to convert certain columns to 32-bit types to save memory (default is False).
        - copy: A boolean indicating whether to create a copy of the input DataFrame before transformation (default is True).
        - group_cols: A list of column names to group by before imputation (default is ['parking_id']).
    """
    def __init__(self, cols_to_impute=['spaces_left'], freq_minutes=5, ffill_limit=60, convert_to_32=False, copy=True, group_cols=['parking_id']):
        self.cols_to_impute = cols_to_impute
        self.freq_minutes = freq_minutes
        self.ffill_limit = ffill_limit
        self.convert_to_32 = convert_to_32
        self.copy = copy
        self.group_cols = group_cols

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        self.group_fill_values_ = X.groupby(self.group_cols)[self.cols_to_impute].median()

        if hasattr(X, "columns"):
            self.feature_names_in_ = X.columns
        return self  

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        if self.copy:
            X = X.copy()
        if not X.index.is_monotonic_increasing:
            X = X.sort_index()
            
        for col in self.cols_to_impute:
            X[f'{col}_is_imputed'] = X[col].isna().astype(int)
            
        X_grouped = X.groupby(self.group_cols)
        #Linear Interpolation leaks data ffill does not
        limit_small = self.ffill_limit // self.freq_minutes
        
        X[self.cols_to_impute] = X_grouped[self.cols_to_impute].ffill(limit=limit_small)

        # Seasonality Fill (7 days ago)
        records_per_week = 7 * 24 * (60 // self.freq_minutes)
        X[self.cols_to_impute] = X[self.cols_to_impute].fillna(
            X_grouped[self.cols_to_impute].shift(records_per_week)
        )

        # Seasonality Fill (28 days ago - 4 weeks)
        records_per_month = 4 * 7 * 24 * (60 // self.freq_minutes)
        X[self.cols_to_impute] = X[self.cols_to_impute].fillna(
            X_grouped[self.cols_to_impute].shift(records_per_month)
        )

        for col in self.cols_to_impute:
            # Map the specific column's median from the fit step
            fill_values = X[self.group_cols[0]].map(self.group_fill_values_[col])
            X[col] = X[col].fillna(fill_values)

        if self.convert_to_32:
            try:
                imputed_flags = [f'{c}_is_imputed' for c in self.cols_to_impute]
                target_cols = self.cols_to_impute + imputed_flags
                
                if 'active_users' in X.columns:
                    X['active_users'] = X['active_users'].astype(np.int32) # convert it back to int as active user should no longer have NaN values

                if 'spaces_left' in X.columns:
                    X['spaces_left'] = X['spaces_left'].astype(np.int32) # convert spaces left to int as it should no longer have NaN values after imputation

                float_cols = X[target_cols].select_dtypes(include=['float64']).columns
                int_cols = X[target_cols].select_dtypes(include=['int64']).columns
                X[float_cols] = X[float_cols].astype(np.float32)
                X[int_cols] = X[int_cols].astype(np.int32)
            except Exception as e:
                print(f"Error occurred while converting data types: {e}")

        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        
        # Add the new flag columns to the output list
        new_flags = [f'{col}_is_imputed' for col in self.cols_to_impute]
        return list(input_features) + new_flags

In [ ]:
from preprocessing.parking_imputer import ParkingImputer

In [ ]:
parking_imputer = ParkingImputer(cols_to_impute=['spaces_left','active_users','sks_dist_to_users_ratio'], freq_minutes=CONST_FREQUENCY_MINUTES, ffill_limit=60, convert_to_32=CONST_CONVERT_32_BIT_VALUES)
df_inputed = parking_imputer.fit_transform(df_megred_calendar)
df_inputed.head(10)

In [ ]:
plot_parking_trends(df_inputed, df_parkings, plotted_value='sks_dist_to_users_ratio')

In [ ]:
plot_parking_trends(df_inputed, df_parkings, plotted_value='spaces_left')

In [ ]:
expansions = {
    "Parking Wrońskiego": '2025-05-21',
    #Others are ommited so they will be treated as never expanded (add them if they were expanded)
}
size_before_expansion ={
    "Parking Wrońskiego": 192,
    #Others are ommited so they will be treated as never expanded (add them if they were expanded)
}

# Capcity corrector
This class corrects negative spaces left reported in dataframe and adds flags for anomalous data. 
Also, it calculates the utilization rate for parkings to reflect changes in capacity over time. 

In [ ]:
%%writefile feature_engineering/parking_capacity_corrector.py 
import pandas as pd
import numpy as np 
import datetime as dt
from sklearn.base import BaseEstimator, TransformerMixin
class ParkingSpacesCorrector(BaseEstimator, TransformerMixin):
    """Corrects parking space counts based on expansion status and calculates additional features related to capacity and utilization.
    First grouping columns should be parking_id as it is used for the mapping of expansion dates and previous capacities.
     Parameters:
     - parkings_df: A DataFrame containing parking information with 'id', 'name', and 'max_spaces_left' columns.
     - expansion_date_map: A dictionary mapping parking names to their expansion dates (e.g., {"Parking Wrońskiego": '2025-05-21'}).
     - previous_size_map: A dictionary mapping parking names to their capacities before expansion (e.g., {"Parking Wrońskiego": 192}).
     - convert_to_32: A boolean indicating whether to convert certain columns to 32-bit types to save memory (default is False).
     - copy: A boolean indicating whether to create a copy of the input DataFrame before transformation (default is True).
     - group_cols: A list of column names to group by before applying the corrections (default is ['parking_id']).
    """
    
    def __init__(self, parkings_df, expansion_date_map, previous_size_map, convert_to_32=False, copy = True, group_cols=['parking_id']):
        self.parkings_df = parkings_df
        self.expansion_date_map = expansion_date_map
        self.previous_size_map = previous_size_map
        self.convert_to_32 = convert_to_32
        self.copy = copy
        self.group_cols = group_cols
        if not isinstance(parkings_df, pd.DataFrame):
            raise TypeError("This transformer requires parkings_df to be a pandas DataFrame, not a numpy array.")

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        if hasattr(X, "columns"):
                self.feature_names_in_ = X.columns

        # Pre-calculate mapping data once during fit
        meta = self.parkings_df[['id', 'name', 'max_spaces_left']].copy()
        
        # Map expansion dates and previous capacities
        meta['exp_date'] = meta['name'].map(self.expansion_date_map).fillna("2200-01-01")
        meta['cap_before'] = meta['name'].map(self.previous_size_map).fillna(meta['max_spaces_left'])
        
        # Create a fast lookup dataframe indexed by 'id' 
        self.parkings_meta_ = meta.set_index('id')[['exp_date', 'cap_before', 'max_spaces_left']]
        return self  

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        if self.copy:
            X = X.copy()

        X['overbooked'] = (X['spaces_left'] < 0).astype(int)
        X['overbooked_spaces'] = np.where(X['overbooked'] == 1, -X['spaces_left'], 0)

        X = X.join(self.parkings_meta_, on=self.group_cols[0])

        X['is_expanded'] = (X['measured_at'] >= X['exp_date']).astype(int)
        
        # Capacity assignment
        X['total_capacity'] = np.where(X['is_expanded'] == 1, X['max_spaces_left'], X['cap_before'])
        
        # Clip spaces_left to be between 0 and total_capacity
        X['spaces_left'] = X['spaces_left'].clip(lower=0, upper=X['total_capacity'])

        # Calculate utilization rate
        X['utilization_rate'] = (X['total_capacity'] - X['spaces_left']) / X['total_capacity']

        # Clean up temporary columns
        X = X.drop(columns=['exp_date', 'cap_before', 'max_spaces_left'])

        if self.convert_to_32:
            try:
                int_cols = ['overbooked', 'is_expanded', 'overbooked_spaces', 'total_capacity']
                float_cols = ['utilization_rate']
                X[int_cols] = X[int_cols].astype(np.int32)
                X[float_cols] = X[float_cols].astype(np.float32)
            except Exception as e:
                print(f"Error occurred while converting data types: {e}")

        return X
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        return list(input_features) + ['overbooked', 'overbooked_spaces', 'is_expanded', 'total_capacity', 'utilization_rate']

In [ ]:
from feature_engineering.parking_capacity_corrector import ParkingSpacesCorrector


In [ ]:
parkings_capacity_corrector = ParkingSpacesCorrector(df_parkings, expansions, size_before_expansion, convert_to_32=CONST_CONVERT_32_BIT_VALUES)
df_corrected = parkings_capacity_corrector.fit_transform(df_inputed)

df_corrected[['spaces_left', 'total_capacity', 'utilization_rate', 'overbooked']].head(10)

In [ ]:
df_parkings.head()

In [ ]:
# Juse me goofing around
# dict_test ={
#     "Parking Wrońskiego": "Open_for_everyone",
#     "Architektura": "Open_for_university_workewrs_only",
#     "Polinka": "Open_for_everyone_for_university_workewrs_and_PHD_students_only",
#     "D20 - D21": "Open_for_everyone",
#     "GEO LO1 Geocentrum": "Open_for_everyone"
# }

# x_copied = df_parkings[['id', 'name']].copy()
# x_copied['open_for_who'] = x_copied['name'].map(dict_test)

# x_copied.head()
# x_copied = x_copied.set_index('id')
# df_parkings = df_parkings.join(x_copied['open_for_who'], on='id')
# df_parkings.head()

In [ ]:
plot_parking_trends(df_corrected, df_parkings, plotted_value='utilization_rate')

# Day features - creator
It adds basic day related features to dataframe like day of the week and is_weekend flag. Also it adds is_open flag based on metadata stored in df_parkings. 

In [ ]:
%%writefile feature_engineering/day_features.py 
import pandas as pd
import numpy as np 
from sklearn.base import BaseEstimator, TransformerMixin
"""Adds day-related features and parking open status based on parking hours.
Parameters:
- df_parkings: A DataFrame containing parking information with 'id', 'open_hour', and 'close_hour' columns.
- convert_to_32: A boolean indicating whether to convert the 'is_open' column to 32-bit integer type to save memory (default is False).    
"""
class DayFeaturesCreator(BaseEstimator, TransformerMixin):
    def __init__(self, df_parkings, convert_to_32=False, copy=True):
        self.df_parkings = df_parkings.copy()
        self.convert_to_32 = convert_to_32
        self.copy = copy

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
                self.feature_names_in_ = X.columns
        return self  

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        if self.copy:
            X = X.copy()

        parking_info = self.df_parkings[['id', 'open_hour', 'close_hour']]
        #check if columns required to merge exist
        try: 
            parking_subset = X[['parking_id']]
        except KeyError as e:
            raise KeyError(f"Missing required columns in input X. Expected 'parking_id'. Original error: {e}")

        try:
            parking_info_subset = parking_info[['id']]
        except KeyError as e:
            raise KeyError(f"Missing required columns in parking_info. Expected 'id'. Original error: {e}")

        #Temporary merge to get open/close hours for each parking_id
        X_merged = X.merge(parking_info, left_on='parking_id', right_on='id', how='left')

        current_time_str = X['measured_at'].dt.strftime('%H:%M:%S')
        
        # Ensure start/end are strings for comparison
        open_time = X_merged['open_hour'].astype(str)
        close_time = X_merged['close_hour'].astype(str)

        #"Is Open" Logic
        is_open_standard = (current_time_str >= open_time) & (current_time_str <= close_time)
        
        # Always open logic
        always_open = (open_time == '00:00:00') & (close_time == '00:00:00')

        # Combine logic: Open if standard rule applies OR if it's a 24/7 lot
        X['is_open'] = (is_open_standard | always_open).astype(int)

        # Fill NaNs with 0 (closed) just in case a parking_id didn't match
        X['is_open'] = X['is_open'].fillna(0).astype(int)

        if self.convert_to_32:
            try:
                numerical_cols = ['is_open']
                X[numerical_cols] = X[numerical_cols].astype(np.int32)
            except Exception as e:
                print(f"Error occurred while converting data types: {e}")
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        return list(input_features) + ['is_open']

In [ ]:
from feature_engineering.day_features import DayFeaturesCreator

In [ ]:
day_feature_creator = DayFeaturesCreator(df_parkings, convert_to_32=CONST_CONVERT_32_BIT_VALUES)
df_with_day_features = day_feature_creator.fit_transform(df_corrected)
df_with_day_features.iloc[81000:81010]

# Rolling features creator
This class takes a dictionary when created. It creates new features based on the configuration passed in the dictionary. 

In [ ]:
# hour every 12 records (1 hour = 12 records at 5 min frequency, 7 days = 288 records per day * 7)
rolling_features = {
    # Key : (Suffix,Type, Window Size, List of Columns)
    # Avaialvble types: "delta", "mean", "std", "max", "min"
    1: ("delta_1h", "delta", 12, ["spaces_left"]), 
    2: ("Rolling_mean_1d", "mean", 288, ["spaces_left"]), 
    3: ("Rolling_mean_7d", "mean", 288*7, ["spaces_left"]),
    4: ("Rolling_std_1d", "std", 288, ["spaces_left"]),
    5: ("Rolling_std_7d", "std", 288*7, ["spaces_left"]),
    6: ("Rolling_max_7d", "max", 288*7, ["spaces_left"]),
    7: ("Rolling_min_7d", "min", 288*7, ["spaces_left"]),
}

In [ ]:
%%writefile feature_engineering/rolling_features_creator.py 
import pandas as pd
import numpy as np 
import datetime as dt
from sklearn.base import BaseEstimator, TransformerMixin
class RollingFeaturesCreator(BaseEstimator, TransformerMixin):
    """Creates rolling features such as rolling mean, std, max, min, and deltas for specified columns based on parking_id groups.
     Parameters:
     - rolling_features: A dictionary specifying the rolling features to create, where each key is an identifier and the value is a tuple containing (suffix, feature_type, window_size, columns).
     - convert_to_32: A boolean indicating whether to convert the new feature columns to 32-bit types to save memory (default is False).
     - shift_guard_periods: The number of periods to shift the data before calculating rolling features to prevent data leakage (default is 1 period, which corresponds to 5 minutes if the frequency is 5 minutes).
     - copy: A boolean indicating whether to create a copy of the input DataFrame before transformation (default is True). 
     - group_cols: A list of column names to group by before calculating rolling features (default is ['parking_id']).
     - fill_nan_with_zero: A boolean indicating whether to fill NaN values in the new feature columns with zero (default is False).
    """
    def __init__(self, rolling_features, convert_to_32=False, shift_guard_periods=1, copy = True, group_cols=['parking_id'], fill_nan_with_zero=False):
        self.rolling_features = rolling_features
        self.group_cols = group_cols
        self.convert_to_32 = convert_to_32
        self.shift_guard_periods = shift_guard_periods
        self.copy = copy
        self.fill_nan_with_zero = fill_nan_with_zero

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = X.columns
        return self  

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        if self.copy:
            X = X.copy()
        

        grouped = X.groupby(self.group_cols)    
        
        for _, (suffix, feature_type, window_size, columns) in self.rolling_features.items():
            for col in columns:
                new_col_name = f"{col}_{suffix}"

                X[new_col_name] = grouped[col].shift(self.shift_guard_periods)
                
                if feature_type == "delta":
                    X[new_col_name] = grouped[new_col_name].diff(window_size)
                
                else:
                    X[new_col_name] = (
                        grouped[new_col_name]
                        .rolling(window=window_size, min_periods=1)
                        .agg(feature_type)
                        .reset_index(level=0, drop=True) 
                    )

                if self.fill_nan_with_zero:
                    X[new_col_name] = X[new_col_name].fillna(0)

        if self.convert_to_32:
            try:
                new_cols = [f"{col}_{suffix}" for _, (suffix, _, _, columns) in self.rolling_features.items() for col in columns]
                int_cols = X[new_cols].select_dtypes(include=['int64']).columns
                float_cols = X[new_cols].select_dtypes(include=['float64']).columns
                X[int_cols] = X[int_cols].astype(np.int32)
                X[float_cols] = X[float_cols].astype(np.float32)
            except Exception as e:
                print(f"Error occurred while converting data types: {e}")
            
        return X
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        new_cols = [f"{col}_{suffix}" for _, (suffix, _, _, columns) in self.rolling_features.items() for col in columns]
        return list(input_features) + new_cols

In [ ]:
from feature_engineering.rolling_features_creator import RollingFeaturesCreator

In [ ]:
rolling_features_creator = RollingFeaturesCreator(rolling_features =rolling_features, convert_to_32=CONST_CONVERT_32_BIT_VALUES, shift_guard_periods=CONST_SHIFT_GUARD_PERIODS)
df_with_rolling_features = rolling_features_creator.fit_transform(df_with_day_features)
df_with_rolling_features.iloc[81000:81010]

# Expanding features creator
Creates expanding features based on configuration passed as a dictionary upon class creation. 

In [ ]:
expanding_features = {
    # Key : (Suffix, Type, List of Columns)
    1: ("cumulative_mean", "mean", ["spaces_left"]),
    2: ("cumulative_max", "max", ["spaces_left"]),
    3: ("cumulative_min", "min", ["spaces_left"]),
    4: ("cumulative_trend", "std", ["spaces_left"]),
}

In [ ]:
%%writefile feature_engineering/expanding_features_creator.py
import pandas as pd
import numpy as np 
import datetime as dt
from sklearn.base import BaseEstimator, TransformerMixin
class ExpandingFeaturesCreator(BaseEstimator, TransformerMixin):
    """Creates expanding features such as cumulative mean, max, min, and trend for specified columns based on parking_id groups.
     Parameters:
     - expanding_features: A dictionary specifying the expanding features to create, where each key is an identifier and the value is a tuple containing (suffix, feature_type, columns).
     - convert_to_32: A boolean indicating whether to convert the new feature columns to 32-bit types to save memory (default is False).
     - shift_guard_periods: The number of periods to shift the data before calculating expanding features to prevent data leakage (default is 1 period, which corresponds to 5 minutes if the frequency is 5 minutes).
     - copy: A boolean indicating whether to create a copy of the input DataFrame before transformation (default is True). 
     - group_cols: A list of column names to group by before calculating expanding features (default is ['parking_id']).
     - fill_nan_with_zero: A boolean indicating whether to fill NaN values in the new feature columns with zero (default is False).
    """
    def __init__(self, expanding_features, convert_to_32=False, shift_guard_periods=1, copy=True, group_cols=['parking_id'], fill_nan_with_zero=False):
        self.expanding_features = expanding_features
        self.group_cols = group_cols
        self.convert_to_32 = convert_to_32
        self.copy = copy
        self.shift_guard_periods = shift_guard_periods
        self.fill_nan_with_zero = fill_nan_with_zero

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = X.columns
        
        return self  

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        if self.copy:
            X = X.copy()
        
        for _, (suffix, feature_type, columns) in self.expanding_features.items():
            for col in columns:
                new_col_name = f"{col}_{suffix}"
                

                X[new_col_name] = (
                    X.groupby(self.group_cols)[col]
                     .expanding()
                     .agg(feature_type)   # Uses "mean", "max", "min", "std" dynamically
                     .groupby(level=0) 
                     .shift(self.shift_guard_periods)
                     .reset_index(level=0, drop=True)
                )
                if self.fill_nan_with_zero:
                    X[new_col_name] = X[new_col_name].fillna(0)

        if self.convert_to_32:
            try:    
                new_cols = [f"{col}_{suffix}" for _, (suffix, _, columns) in self.expanding_features.items() for col in columns]
                int_cols = X[new_cols].select_dtypes(include=['int64']).columns
                float_cols = X[new_cols].select_dtypes(include=['float64']).columns
                X[int_cols] = X[int_cols].astype(np.int32)
                X[float_cols] = X[float_cols].astype(np.float32)
            except Exception as e:
                print(f"Error occurred while converting data types: {e}")
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        
        new_feature_names = []
        for _, (suffix, _, columns) in self.expanding_features.items():
            for col in columns:
                new_feature_names.append(f"{col}_{suffix}")
        
        return list(input_features) + new_feature_names

In [ ]:
from feature_engineering.expanding_features_creator import ExpandingFeaturesCreator

In [ ]:
expanding_features_creator = ExpandingFeaturesCreator(expanding_features, convert_to_32=CONST_CONVERT_32_BIT_VALUES, shift_guard_periods=CONST_SHIFT_GUARD_PERIODS)
df_with_expanding_features = expanding_features_creator.fit_transform(df_with_rolling_features)

# Lag feature creator
Takes config in dictionary form and performs lag on specified columns corresponding to specified lag time. 

In [ ]:
# hour every 12 records (1 hour = 12 records at 5 min frequency, 7 days = 288 records per day * 7)
lag_features = {
    # Key : (Suffix, Window Size, List of Columns)
    1: ("lag_1h", 12, ["spaces_left", "utilization_rate"]), 
    2: ("lag_7d", 288*7, ["spaces_left", "utilization_rate", "spaces_left_Rolling_mean_7d", "spaces_left_Rolling_std_7d", "spaces_left_Rolling_max_7d", "spaces_left_Rolling_min_7d"]), 
}

In [ ]:
%%writefile feature_engineering/lag_features_creator.py 
import pandas as pd
import numpy as np 
import datetime as dt
from sklearn.base import BaseEstimator, TransformerMixin
class LagFeaturesCreator(BaseEstimator, TransformerMixin):
    """Creates lag features for specified columns based on parking_id groups
     Parameters:
     - lag_features: A dictionary specifying the lag features to create, where each key is an identifier and the value is a tuple containing (suffix, window_size, columns).
     - convert_to_32: A boolean indicating whether to convert the new feature columns to 32-bit types to save memory (default is False).
     - copy: A boolean indicating whether to create a copy of the input DataFrame before transformation (default is True). 
     - group_cols: A list of column names to group by before calculating lag features (default is ['parking_id']).
     - fill_nan_with_zero: A boolean indicating whether to fill NaN values in the new feature columns with zero (default is False).
    """
    def __init__(self, lag_features, convert_to_32=False, copy=True, group_cols=['parking_id'], fill_nan_with_zero=False):
        self.lag_features = lag_features
        self.group_cols = group_cols
        self.convert_to_32 = convert_to_32  
        self.copy = copy
        self.fill_nan_with_zero = fill_nan_with_zero

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = X.columns
        return self   

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        if self.copy:
            X = X.copy()
        
        for _, (suffix, window_size, columns) in self.lag_features.items():
            
            new_col_names = [f"{col}_{suffix}" for col in columns]
            X[new_col_names] = X.groupby(self.group_cols)[columns].shift(window_size)
            
            # Handle Missing Values
            if self.fill_nan_with_zero:
                X[new_col_names] = X[new_col_names].fillna(0)

        if "utilization_rate" in X.columns:
            X = X.drop(["utilization_rate"], axis=1)  # Drop original utilization_rate to prevent data leakage, as we have its lagged versions as features
        if self.convert_to_32:
            try:
                new_cols = [f"{col}_{suffix}" for _, (suffix, _, columns) in self.lag_features.items() for col in columns]
                int_cols = X[new_cols].select_dtypes(include=['int64']).columns
                float_cols = X[new_cols].select_dtypes(include=['float64']).columns
                X[int_cols] = X[int_cols].astype(np.int32)
                X[float_cols] = X[float_cols].astype(np.float32)
            except Exception as e:
                print(f"Error occurred while converting data types: {e}")
                
        return X
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        
        new_feature_names = []
        for _, (suffix, _, columns) in self.lag_features.items():
            for col in columns:
                new_feature_names.append(f"{col}_{suffix}")
        
        return list(input_features) + new_feature_names

In [ ]:
from feature_engineering.lag_features_creator import LagFeaturesCreator

In [ ]:
lag_features_creator = LagFeaturesCreator(lag_features, convert_to_32=CONST_CONVERT_32_BIT_VALUES)
df_with_lags = lag_features_creator.fit_transform(df_with_expanding_features)


df_with_lags.iloc[80000:80010]

In [ ]:
print(df_with_lags.memory_usage(deep=True))

# To get the total memory of the entire DataFrame in Megabytes (MB):
total_mb =  df_with_lags.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Total Memory: {total_mb:.2f} MB")

# Cannibalization feature creator
This uses relations in distance between parkings to add cannibalization features based on the configuration specified in the dictionary. 

In [ ]:
cannibalistion_features = {
    # Key : (Suffix, Type, List of Columns)
    1: ("cann_closest_parking", "closest", ["spaces_left", "spaces_left_lag_7d", "utilization_rate_lag_1h", "utilization_rate_lag_7d", "spaces_left_delta_1h"]),
    2: ("cann_second_closest_parking", "second_closest", ["spaces_left", "utilization_rate_lag_1h"]),
    3: ("cann_third_closest_parking", "third_closest", ["spaces_left", "utilization_rate_lag_1h"]),
    4: ("cann_fourth_closest_parking", "fourth_closest", ["spaces_left", "utilization_rate_lag_1h"]),
}

Dataframe converted to float32 occupies only 55% of space of orginal data. This is a wothwile optimalisation althrough even unconverted data frame is only shigty above 100MB witch isn't unmanagable. 

In [ ]:
%%writefile feature_engineering/cannibalisation_adder.py 
import pandas as pd
import numpy as np 
from sklearn.base import BaseEstimator, TransformerMixin

class CannibalisationFeaturesCreator(BaseEstimator, TransformerMixin):
    """Adds cannibalisation features based on the nearest neighboring parkings' data, with a guard period to prevent data leakage.
     Parameters:
     - parkings_df: A DataFrame containing parking information with 'id' and neighbor parking columns (e.g., 'closest_id', 'second_closest_id', etc.).
     - cannibalisation_features: A dictionary specifying the cannibalisation features to create, where each key is an identifier and the value is a tuple containing (suffix, neighbor_type, columns).
     - cannibalisation_guard: The number of periods to shift the neighbor data to prevent data leakage (default is 12 periods, which corresponds to 1 hour if the frequency is 5 minutes).
     - convert_to_32: A boolean indicating whether to convert the new feature columns to 32-bit types to save memory (default is False).
     - copy: A boolean indicating whether to create a copy of the input DataFrame before transformation (default is True). 
     - group_cols: A list of column names to group by before applying the cannibalisation features (default is ['parking_id']).
     - fill_nan_with_zero: A boolean indicating whether to fill NaN values in the new feature columns with zero (default is False).
    """
    def __init__(self, parkings_df, cannibalisation_features, cannibalisation_guard=12, convert_to_32=False, copy=True, group_cols=['parking_id'], fill_nan_with_zero=False):
        self.parkings_df = parkings_df.copy()
        self.cannibalisation_features = cannibalisation_features
        self.cannibalisation_guard = cannibalisation_guard
        self.convert_to_32 = convert_to_32
        self.copy = copy
        self.fill_nan_with_zero = fill_nan_with_zero
        self.group_cols = group_cols

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = X.columns
        return self   

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("This transformer requires X to be a pandas DataFrame, not a numpy array.")
        if self.copy:
            X_out = X.copy()
        else:
            X_out = X
        
        required_neighbor_map = {} 
        for _, (_, neighbor_type, _) in self.cannibalisation_features.items():
            col_name = f"{neighbor_type}_id"
            required_neighbor_map[neighbor_type] = col_name

        cols_to_merge = list(set(required_neighbor_map.values()))
        
        #check if columns required to merge exist 
        try:
            parkings_subset = self.parkings_df[['id'] + cols_to_merge]
        except KeyError as e:
            raise KeyError(f"Missing required columns in parkings_df. Expected 'id' and {cols_to_merge}. Original error: {e}")
        # Merge metadata (Left join to keep X structure)
        X_merged = X_out.merge(
            self.parkings_df[['id'] + cols_to_merge],
            left_on='parking_id',
            right_on='id',
            how='left'
        ).drop(columns=['id'])

        for key, (suffix, neighbor_type, columns) in self.cannibalisation_features.items():
            
            neighbor_col_name = required_neighbor_map.get(neighbor_type)
            
            if neighbor_col_name not in X_merged.columns:
                continue

            # Prepare the "Lookup" table (The neighbor data)
            lookup_df = X[['measured_at', 'parking_id'] + columns].copy()
            
            # Apply Cannibalisation Guard (Latency Simulation)
            if self.cannibalisation_guard <= 0:
                self.cannibalisation_guard = 12  # Default to 1 hour if invalid value is provided
                raise ValueError("Cannibalisation guard must be a positive integer representing the number of periods to shift.")

            if self.cannibalisation_guard > 0:
                lookup_df = lookup_df.sort_values('measured_at')
                lookup_df[columns] = lookup_df.groupby(self.group_cols)[columns].shift(self.cannibalisation_guard)

            # Rename columns for the merge
            rename_dict = {col: f"{col}_{suffix}" for col in columns}
            rename_dict['parking_id'] = neighbor_col_name
            
            lookup_df = lookup_df.rename(columns=rename_dict)


            # Merge
            X_merged = X_merged.merge(
                lookup_df,
                on=['measured_at', neighbor_col_name],
                how='left'
            )
            
            # Fill NaNs created by the shift or missing neighbors
            new_cols = [f"{col}_{suffix}" for col in columns]
            if self.fill_nan_with_zero:
                X_merged[new_cols] = X_merged[new_cols].fillna(0)

        X_merged = X_merged.drop(columns=cols_to_merge, errors='ignore')

        if self.convert_to_32:
            all_new_cols = []
            for _, (suffix, _, columns) in self.cannibalisation_features.items():
                all_new_cols.extend([f"{col}_{suffix}" for col in columns])
            try:
                cols_to_convert = [c for c in all_new_cols if c in X_merged.columns]
                int_cols = X_merged[cols_to_convert].select_dtypes(include=['int64']).columns
                float_cols = X_merged[cols_to_convert].select_dtypes(include=['float64']).columns
                X_merged[int_cols] = X_merged[int_cols].astype(np.int32)
                X_merged[float_cols] = X_merged[float_cols].astype(np.float32)
            except Exception as e:
                print(f"Error occurred while converting data types: {e}")   

        return X_merged

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        
        new_feature_names = []
        for _, (suffix, _, columns) in self.cannibalisation_features.items():
            for col in columns:
                new_feature_names.append(f"{col}_{suffix}")
        
        return list(input_features) + new_feature_names

In [ ]:
from feature_engineering.cannibalisation_adder import CannibalisationFeaturesCreator    

In [ ]:
cannibalistion_features_creator = CannibalisationFeaturesCreator(df_parkings, cannibalistion_features, cannibalisation_guard=CONST_CANNIVALISATION_GUARD_PERIODS, convert_to_32=CONST_CONVERT_32_BIT_VALUES)
df_with_cannibalisation = cannibalistion_features_creator.fit_transform(df_with_lags)
df_with_cannibalisation.iloc[80000:80010]

In [ ]:
print(df_with_cannibalisation.memory_usage(deep=True))

# To get the total memory of the entire DataFrame in Megabytes (MB):
total_mb =  df_with_cannibalisation.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Total Memory: {total_mb:.2f} MB")

In [ ]:

# 1. Calculate and sort the correlation
# We drop 'spaces_left' so we don't plot its 1.0 correlation with itself
corr_series = df_with_cannibalisation.corr(method='pearson')['spaces_left'].sort_values(ascending=False)
corr_series = corr_series.drop('spaces_left')

# 2. Create the Diverging Bar Chart
plt.figure(figsize=(10, 8))

# Define colors: Green for positive, Red for negative
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in corr_series.values]

# Plotting
bars = plt.barh(corr_series.index, corr_series.values, color=colors, edgecolor='black', alpha=0.8)

# Add a vertical line at zero for balance
plt.axvline(x=0, color='black', linestyle='-', linewidth=1.5)

# Formatting
plt.title('Correlation with "spaces_left"', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Pearson Correlation Coefficient', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Optional: Add text labels on the bars for exact values
for bar in bars:
    width = bar.get_width()
    plt.text(width + (0.01 if width > 0 else -0.01), 
             bar.get_y() + bar.get_height()/2, 
             f'{width:.2f}', 
             va='center', 
             ha='left' if width > 0 else 'right',
             fontsize=10, 
             fontweight='bold')

# Ensure the plot is symmetric around zero
limit = max(abs(corr_series.min()), abs(corr_series.max())) + 0.1
plt.xlim(-limit, limit)

plt.tight_layout()
plt.show()

In [ ]:

# 1. Calculate and sort the correlation
# We drop 'spaces_left' so we don't plot its 1.0 correlation with itself
corr_series = df_with_cannibalisation.corr(method='spearman')['spaces_left'].sort_values(ascending=False)
corr_series = corr_series.drop('spaces_left')

# 2. Create the Diverging Bar Chart
plt.figure(figsize=(10, 8))

# Define colors: Green for positive, Red for negative
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in corr_series.values]

# Plotting
bars = plt.barh(corr_series.index, corr_series.values, color=colors, edgecolor='black', alpha=0.8)

# Add a vertical line at zero for balance
plt.axvline(x=0, color='black', linestyle='-', linewidth=1.5)

# Formatting
plt.title('Correlation with "spaces_left"', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Pearson Correlation Coefficient', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Optional: Add text labels on the bars for exact values
for bar in bars:
    width = bar.get_width()
    plt.text(width + (0.01 if width > 0 else -0.01), 
             bar.get_y() + bar.get_height()/2, 
             f'{width:.2f}', 
             va='center', 
             ha='left' if width > 0 else 'right',
             fontsize=10, 
             fontweight='bold')

# Ensure the plot is symmetric around zero
limit = max(abs(corr_series.min()), abs(corr_series.max())) + 0.1
plt.xlim(-limit, limit)

plt.tight_layout()
plt.show()